# Logistic regression

Here is the implementation of my first model to recommend videos! The aim of this notebook is to implement a logistic regression model that predict videos with a content based approch. I wanted to build a very simple model to find out if a very simple model as this one can make good predictions. I was first implementing this model to use it as a baseline for other content based models that I may implement, but it turns out that it was not bad at all... (we will see later!)

For each models I will always describe here my inputs/outputs to get a better intuition of what feed the models and how is it possible to make predictions. So here is the inputs:

- user engineered features: average notes by video tags (recall embeddings section when we built . it!). This matrix is called `user_avg_ratings` in the code. It contains the `user_id` following the 31 average ratings (because there are 31 video tags!)
- video content features: first there is the video duration following the one hot encoding of their respective tags

We then concatenate those vectors to create an input matrix that will feed our logistic regression. On each line, we will have a combination of a user and a video if and only if the user has already watched the video. 

On the other hand, the outputs are the vector of like/dislike that is each line maps to the input vector that result the amount of time the user spent on watching the video on the same line. Like/dislike here is 0 or 1 according to the watch ratio (recall embeddings section where we define the threshold of watch ratio to 0.8!).

To predict recommendations, the aim is to concat every videos embeddings with the one of one single user to predict which are the videos with the highest probabilities that the user may like it.

## Import libraries

We will use here the implementation of `scikit learn` to implement the logistic regression.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import pandas as pd
import sys
sys.path.append('../')
import metrics

MANUAL_RANDOM_SEED=42
EMBEDDINGS_PATH='../embeddings'

## Import embeddings

Let's use our embeddings by loading them using Pandas.

In [2]:
videos_content = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_content.pkl')
videos_tags = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_one_hot_tags.pkl')
user_avg_ratings = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_avg_ratings.pkl')
user_ratings = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_ratings.pkl')

In [3]:
# Checking shape is always important for me to visualize in my head better!
videos_content.shape, videos_tags.shape, user_avg_ratings.shape, user_ratings.shape

((9937, 9), (9937, 31), (7176, 31), (10064065, 4))

## Pre-processing intputs/outputs to suit the model

We need to make some tweaks to fit our input and output vectors to suit our needs to implement the logistic regression model. For instance, we only want to keep the `video_duration` field in video content because other are useless in our case (recall how we built embedding of video content).

In [4]:
videos_content.head()

,author_id,music_id,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt,video_duration
video_id,,,,,,,,,
0,3309,3350323409,805839,411691,224751,24493,448,105,5968
1,4978,1812462382,136718,141667,77368,8267,59,64,25264
2,939,0,650377,670248,474834,3180,30,19,8034
3,5889,0,16406,9137,3966,941,7,26,23267
4,4284,3442844592,825,610,280,3,0,0,18228


In [5]:
# Keep only video_duration 
videos_content_full = videos_content
videos_content = videos_content[['video_duration']]
# Remove watch_ration, we use like/dislike between 0 and 1 for logistic regression
user_ratings = user_ratings.drop('watch_ratio', axis=1)

In [6]:
# Concatenating user ratings with user average ratings for each video category (video tags)
X_user = user_ratings.merge(user_avg_ratings, on='user_id')

In [7]:
X_user.head()

,user_id,video_id,like,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,...,avg_rating_tag_21,avg_rating_tag_22,avg_rating_tag_23,avg_rating_tag_24,avg_rating_tag_25,avg_rating_tag_26,avg_rating_tag_27,avg_rating_tag_28,avg_rating_tag_29,avg_rating_tag_30
0,0,3649,1,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0.714286,0.0,1.0,1.0,0.568807,0.556604,0.444444,0.585809,0.75,0.25
1,0,5262,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0.714286,0.0,1.0,1.0,0.568807,0.556604,0.444444,0.585809,0.75,0.25
2,0,6789,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0.714286,0.0,1.0,1.0,0.568807,0.556604,0.444444,0.585809,0.75,0.25
3,0,6812,1,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0.714286,0.0,1.0,1.0,0.568807,0.556604,0.444444,0.585809,0.75,0.25
4,0,183,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0.714286,0.0,1.0,1.0,0.568807,0.556604,0.444444,0.585809,0.75,0.25


In [8]:
# Then merge each ratings with the related video content
X = videos_tags.merge(videos_content, on='video_id')
X = X_user.merge(X, on='video_id')

In [9]:
X.shape

(10064065, 66)

In [10]:
# Here is the fully X input matrix where user content and video content are concatened
# around the user ratings
X.head()

,user_id,video_id,like,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,...,tag_22,tag_23,tag_24,tag_25,tag_26,tag_27,tag_28,tag_29,tag_30,video_duration
0,0,3649,1,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0,0,0,0,0,0,0,0,0,10867
1,0,5262,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0,0,0,1,0,0,0,0,0,7908
2,0,6789,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0,0,0,0,0,0,0,0,0,13267
3,0,6812,1,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0,0,0,0,0,0,1,0,0,10728
4,0,183,0,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,...,0,0,0,0,0,0,1,0,0,6100


## Split train/validation

At this stage, we are using the train test of interactions called `big_matrix` in the KuaiRec dataset. This matrix is a sparse matrix of interactions of all videos and users. When making this model, we want to use those interactions to train our models (here logistic regression). 

Then, after implementing multiple models of the same nature (here logistic regression is used as a content based model), we want to be able to compare those model but not on the final test set (the small matrix that KuaiRec data provides) but on a new validation set. 

So we need to split our train set of ratings! To achieve that, we first retrieve all ratings for each user. Then, 70% of them will go to the test set and put 30% left on the validation one. This enables us to compute our metrics for each user independantly.

Remark: I use the stratify parameter of the scikit-learn's function `train_test_split`. This enable me to keep the distribution of videos according to the likes that is, I will not get surprise of having a validation sets with only liked videos. Without adding this extra parameter, I have the hazard of getting bias in the metrics I measure.

In [11]:
# Group inputs (i.e. ratings) by user
grouped_user = X.groupby('user_id')

# Initialize both train and validation sets
X_train, X_validation = [], []

# Loop over users where data contains all the rows of the current user
for user_id, data in grouped_user:
    # Split user inputs
    user_train, user_validation = train_test_split(data, random_state=MANUAL_RANDOM_SEED, stratify=data['like'], shuffle=True, test_size=0.3)
    X_train.append(user_train)
    X_validation.append(user_validation)

# Convert python lists to dataframes
X_train = pd.concat(X_train)
X_validation = pd.concat(X_validation)

In [12]:
# Get outputs, that is likes for each combination of user/video
y_train = X_train[['like']]
y_validation = X_validation[['like']]

# Remove unused columns that we do want to pass through the model
X_train = X_train.drop(['video_id', 'user_id', 'like'], axis=1)
X_validation = X_validation.drop('like', axis=1)
scaler = StandardScaler().fit(X_train)

In [13]:
X_train.head()

,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,avg_rating_tag_7,avg_rating_tag_8,avg_rating_tag_9,...,tag_22,tag_23,tag_24,tag_25,tag_26,tag_27,tag_28,tag_29,tag_30,video_duration
1178,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,0.542234,0.57346,...,0,0,0,0,0,0,0,0,0,14240
1028,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,0.542234,0.57346,...,0,0,0,0,0,0,0,0,0,35400
911,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,0.542234,0.57346,...,0,0,0,0,1,0,0,0,0,9034
866,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,0.542234,0.57346,...,0,0,0,0,0,0,0,0,0,10500
453,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,0.542234,0.57346,...,0,0,0,0,0,0,1,0,0,5067


In [14]:
y_train.head()

,like
1178,1
1028,0
911,0
866,1
453,1


In [15]:
X_train.shape, X_validation.shape, y_train.shape, y_validation.shape

((7041587, 63), (3022478, 65), (7041587, 1), (3022478, 1))

## Training the model

To train the model, I am going to use the train set and the implementation scikit learn to use logistic regression.

In [22]:
model = LogisticRegression()
model.fit(scaler.transform(X_train), y_train.values.ravel())

LogisticRegression()

## Metrics and evaluation

### A word on accuracy!

Measuring here the accuracy is not a good metric to evaluate our model. Indeed, the classes of liked/disliked videos are inbalanced: we have more disliked videos than liked ones. So accuracy will be biased by this distribution. That is why I will use and show later in the notebook other metrics that I will use for each models. However confusion matrix and precision, recall and f1 score for each class is a good way to get some intuition on how the model performs 

In [23]:
# Unbalanced
y_train.value_counts()

like
0       3795765
1       3245822
Name: count, dtype: int64

In [24]:
# Not a great score because ratings are not equally distributed!
model.score(scaler.transform(X_train), y_train)

0.7234931557332175

In [25]:
# Prediction on the validation set to output the scikit-learn classification report
y_pred = model.predict(scaler.transform(X_validation.drop(['video_id', 'user_id'], axis=1)))

In [26]:
print(confusion_matrix(y_validation, y_pred))
print(classification_report(y_validation, y_pred))

[[1174385  454907]
 [ 379857 1013329]]
              precision    recall  f1-score   support

           0       0.76      0.72      0.74   1629292
           1       0.69      0.73      0.71   1393186

    accuracy                           0.72   3022478
   macro avg       0.72      0.72      0.72   3022478
weighted avg       0.73      0.72      0.72   3022478



In [27]:
# Add back the like column to compute metrics on a single dataframe
validation_set = X_validation.merge(y_validation, right_index=True, left_index=True)
validation_set.head()

,user_id,video_id,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,avg_rating_tag_7,...,tag_23,tag_24,tag_25,tag_26,tag_27,tag_28,tag_29,tag_30,video_duration,like
679,0,5610,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,10280,1
520,0,729,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,1,0,0,11160,0
1664,0,6032,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,7802,1
800,0,2517,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,7634,1
478,0,5626,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,6098,1


### Building the recommendations matrix

To evaluate the model and measure metrics over the validation set, we need to build a recommendation matrix. It contains recommendations for each user. The matrix is sorted by user id and prediction. This enables me to compute different kind of metrics over the same matrix and for different depths (Precision@K implies having to choice K videos for instance).

In [28]:
# Max K, number of video to recommend for each user so that we can compute
# metrics up to K recommendations
max_K = 20
recommendations: pd.DataFrame = None

for user_id in validation_set['user_id'].unique():
    user_data = validation_set[validation_set['user_id'] == user_id]
    y_user = user_data['like']
    X_user = user_data.drop(['like', 'user_id', 'video_id'], axis=1)
    
    # Predict probabilities to get the top videos
    probas = model.predict_proba(scaler.transform(X_user))[:, 1].ravel()
    top_videos = user_data.iloc[probas.argsort()[::-1][:max_K]]

    # Build the recommendations matrix
    if recommendations is None:
        recommendations = top_videos
    else:
        recommendations = pd.concat([recommendations, top_videos])

In [29]:
recommendations.shape

(143520, 66)

In [30]:
recommendations.head()

,user_id,video_id,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,avg_rating_tag_7,...,tag_23,tag_24,tag_25,tag_26,tag_27,tag_28,tag_29,tag_30,video_duration,like
448,0,154,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,3167,1
1165,0,4040,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,3084,1
507,0,5464,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,3270,1
387,0,600,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,3334,1
1949,0,9365,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,0,0,0,0,0,0,4021,1


### Metrics used

I wanted to use suitable metrics to compare models between themselves as well as evaluating the final one recommender algorithm. The goal of a video recommender algorithm such as the ones of YouTube, Tiktok or even Netflix is to provide entertainment engagement and retain the user on the platform as more as possible. Multiple aspect of recommending videos can help achieving this goal:

1. Give user videos the algorithm thinks he/she will like. Even though the algorithm "forget" to recommend popular/interesting videos for the user, we want to maximize as most as possible the confidence of giving great video to the user.
2. The recommender needs to recommend videos that covers multiple topics or trends to avoid giving the same kind of content to the user

First, we will use Precision@K and NDCG@K because of reason `1`. We actually want to be precise in our recommendation even though some great other videos has been put under the radar for a user (that is what Recall@K is measuring). Plus, NDCG enables us to be even more confident in the way we sort videos in the predictions because it gives great importance to liked videos put at the top of the predictions.

Second, I will use the diversity metric to measure how the model predicts a variety of different videos that are not similar. To measure diversity, I will use Inter-List-Diversity@K (ILD@K) to measure how the recommended videos are different from each other (or not!). ILD@K value equals to 0 means videos are really similar otherwise ILD value will be 1. So the goal is to get a ILD@K near 1 as most as possible. To compute diversity, I use video tags and popularity features (such as likes count, shows count,... recall embeddings notebook) so that I can measure how videos are different on both content and popularity aspects.

To compute metrics, we will use the recommendations matrix and use the like column as the gain.

In [31]:
recommendations.shape

(143520, 66)

In [46]:
# Preparing the columns to merge in the recommendations dataframe to compute
# the diversity metric
videos_content_full.columns[2:-1]

Index(['show_cnt', 'play_cnt', 'valid_play_cnt', 'like_cnt', 'comment_cnt',
       'share_cnt'],
      dtype='object')

In [40]:
# Merging video contents with recommendations
recommendations = recommendations.merge(videos_content_full.iloc[:, 2:-1], on='video_id')
recommendations.head()

,user_id,video_id,avg_rating_tag_0,avg_rating_tag_1,avg_rating_tag_2,avg_rating_tag_3,avg_rating_tag_4,avg_rating_tag_5,avg_rating_tag_6,avg_rating_tag_7,...,tag_29,tag_30,video_duration,like,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt
0,0,154,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,3167,1,382427,391206,315349,2201,12,3
1,0,4040,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,3084,1,420771,359851,288843,1397,8,2
2,0,5464,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,3270,1,403791,408138,304236,1516,9,4
3,0,600,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,3334,1,358626,363059,275808,1517,2,1
4,0,9365,0.5,0.544715,0.666667,0.545455,0.448276,0.439252,0.473498,0.591549,...,0,0,4021,1,3151,3185,2059,198,0,0


In [44]:
# Video content columns to compute similarities between videos to compute the diversity metric
video_content_columns = recommendations.columns[33:].tolist()
video_content_columns.remove('like')
print(video_content_columns)

['tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14', 'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21', 'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28', 'tag_29', 'tag_30', 'video_duration', 'show_cnt', 'play_cnt', 'valid_play_cnt', 'like_cnt', 'comment_cnt', 'share_cnt']


In [45]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)
    diversity = metrics.inter_list_diversity_at_k(recommendations, video_content_columns, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print(f'ILD@{k} -> {diversity}')
    print()

Precision@5 -> 0.7970457079152732
NDCG@5 -> 0.9105550966798521
ILD@5 -> 0.8336759025117516

Precision@10 -> 0.7813127090301003
NDCG@10 -> 0.9100079784776821
ILD@10 -> 0.8707895810402027

Precision@15 -> 0.7697881828316611
NDCG@15 -> 0.9090425873726367
ILD@15 -> 0.8930467630124058

Precision@20 -> 0.7592460981047937
NDCG@20 -> 0.9088590977353567
ILD@20 -> 0.9047708720417118



### Interpretation

As we can see, logistic regression has actually did a pretty great job with almost 80% in precision. This model may actually a very good one to use because it is very easy and fast to train. NDGC is super high indicating that top videos are often present in the top of the predictions and the recommended videos are quite diverse in terms of content/popularity! Having a descending ILD value as we use a higher rank K is intuitive as well!